# HMS — all models on identical validation rows

Every prediction is out-of-fold (from the model's fold that never saw that patient). Each model keeps
its own original split; rows are matched by identity (eeg_id + exact vote counts).

| § | What | Why |
|---|---|---|
| 1 | KLD (competition metric) with patient-bootstrap 95% CI, and the paired difference to the reference model | the headline, and whether a difference is real |
| 2 | Brier, class-wise calibration error (ECE), accuracy vs the majority vote, macro-F1, macro-AUROC | other views of the same predictions |
| 3 | Per-class AUROC / recall / precision / F1 | where each model is strong or weak |
| 4 | KLD by number of expert votes | behaviour on easy vs contested samples |
| 5 | Calibration curves | are predicted probabilities as large as the experts' vote shares? |
| 6 | Confusion matrices (row-normalised) | which classes get mixed up |
| 7 | Error correlation between models | which pairs are worth blending |
| 8 | Each model's own folds | stability |

Main subset: rows with ≥10 votes (6,350 rows, VIPEEGNet's evaluation set). Models whose archive also
holds all-row predictions are additionally scored on all 20,183 rows.
Outputs: `results/comparisons/all_models/` (CSV tables, PNG figures, `report.md`).

In [ ]:
PROJECT_DIR = '~/hms'
MODELS = [                                         # [archive tag, display name]; the first one is the reference
    ['vipeegnet_groupkfold_v1', 'VIPEEGNet'],
    ['effnetb0_native_v1',      'EfficientNetB0'],
    ['prism_wf5_v1',            'PRISM'],
    ['prism_wf5_tta_v1',        'PRISM + TTA'],
    ['blend_vip_prism_wf5_v1',  'VIPEEGNet + PRISM blend'],
]
N_BOOT = 5000
OUT_NAME = 'all_models'

In [ ]:
import os, sys, json, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score
P = os.path.abspath(os.path.expanduser(PROJECT_DIR))
sys.path.insert(0, f'{P}/results'); os.environ['HMS_RESULTS'] = f'{P}/results'
from load_results import load_oof, kld
CLASS = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
OUTD = f'{P}/results/comparisons/{OUT_NAME}'; os.makedirs(OUTD, exist_ok=True)
pid = pd.read_csv(f'{P}/hms-data/train.csv', usecols=['eeg_id', 'patient_id']).groupby('eeg_id').patient_id.first()
NAMES = [n for _, n in MODELS]; REF = NAMES[0]
# chart style: fixed categorical order (validated palette), recessive axes, text in ink colours
COL = dict(zip(NAMES, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#6250d6', '#008300', '#e34948']))
SURF, INK, INK2, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF, 'font.size': 10,
                     'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
                     'grid.linewidth': 0.6, 'axes.axisbelow': True, 'legend.frameon': False})

def keyed(tag, subset):
    yt, yp, m = load_oof(tag, subset)
    raw = np.rint(yt * m.votes.values[:, None]).astype(int)
    m['key'] = m.eeg_id.astype(str) + ':' + pd.Series([','.join(map(str, r)) for r in raw]).values
    assert m.key.is_unique, f'{tag}: duplicate row keys'
    return yt, yp, m

def align(subset):
    base = None; preds = {}; folds = {}
    for tag, name in MODELS:
        try: yt, yp, m = keyed(tag, subset)
        except AssertionError as e:
            if subset == 'all': print(f'  ({name}: no all-row predictions, skipped on all rows)'); continue
            raise
        if base is None: base = (yt, m.reset_index(drop=True)); order = np.arange(len(m))
        else:
            assert set(m.key) == set(base[1].key), f'{name}: scored on different {subset} rows than {REF}'
            order = pd.Series(np.arange(len(m)), index=m.key)[base[1].key].values
            assert np.allclose(yt[order], base[0], atol=1e-5), f'{name}: labels differ'
        preds[name] = np.clip(yp[order], 1e-7, 1); preds[name] /= preds[name].sum(1, keepdims=True)
        folds[name] = m.fold.values[order]
    yt, m = base; m['patient_id'] = m.eeg_id.map(pid).values
    return yt, m, preds, folds

D = {}
for subset in ('hq', 'all'):
    D[subset] = align(subset)
    yt, m, preds, _ = D[subset]
    print(f'{subset}: {len(yt):,} rows | {m.patient_id.nunique():,} patients | models: {list(preds)}')

## 1. KLD with patient-level 95% CI, and difference to the reference

In [ ]:
rng = np.random.default_rng(0)
yt, m, preds, folds = D['hq']
up, inv = np.unique(m.patient_id.values, return_inverse=True); cnt = np.bincount(inv)
BS = rng.integers(0, len(up), (N_BOOT, len(up)))                 # same patient resamples for every model -> paired
def boot(x):
    s = np.bincount(inv, x); return s[BS].sum(1) / cnt[BS].sum(1)
K = {n: kld(yt, p) for n, p in preds.items()}
rows = []
for n in NAMES:
    if n not in K: continue
    b = boot(K[n]); d = boot(K[n] - K[REF])
    rows.append({'model': n, 'KLD': K[n].mean(), 'CI low': np.percentile(b, 2.5), 'CI high': np.percentile(b, 97.5),
                 f'vs {REF}': K[n].mean() - K[REF].mean(), 'diff CI low': np.percentile(d, 2.5),
                 'diff CI high': np.percentile(d, 97.5), 'P(better than ref)': (d < 0).mean(),
                 'rows better %': (K[n] < K[REF]).mean() * 100})
kt = pd.DataFrame(rows).set_index('model')
ya, ma, pa, _ = D['all']
kt['KLD all rows'] = pd.Series({n: kld(ya, p).mean() for n, p in pa.items()})
kt.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 0.55 * len(kt) + 1.4))
y = np.arange(len(kt))[::-1]
for yi, (n, r) in zip(y, kt.iterrows()):
    ax.plot([r['CI low'], r['CI high']], [yi, yi], color=COL[n], lw=2, solid_capstyle='round')
    ax.plot(r.KLD, yi, 'o', ms=9, color=COL[n], mec=SURF, mew=2)
    ax.text(r['CI high'] + 0.003, yi, f'{r.KLD:.4f}', va='center', color=INK, fontsize=9.5, zorder=5,
            bbox=dict(fc=SURF, ec='none', pad=1.5))
ax.axvline(kt.loc[REF, 'KLD'], color=INK2, lw=1, ls=(0, (3, 3)))
ax.set_yticks(y, kt.index); ax.tick_params(axis='y', colors=INK, length=0); ax.grid(axis='y', visible=False)
ax.set_xlabel('KL divergence on rows with ≥10 votes (lower is better) · bars = 95% patient-bootstrap CI')
ax.set_title(f'Competition metric, {len(yt):,} identical rows · dashed line = {REF}', loc='left', fontsize=11, color=INK)
lo_, hi_ = kt['CI low'].min(), kt['CI high'].max(); ax.set_xlim(lo_ - 0.01, hi_ + 0.04)
fig.tight_layout(); fig.savefig(f'{OUTD}/fig1_kld.png', dpi=150); plt.close(fig); display(Image(f'{OUTD}/fig1_kld.png'))

## 2. Other overall metrics (≥10-vote rows)

In [ ]:
yti = yt.argmax(1); oh = (yti[:, None] == np.arange(6)).astype(int)
def ece_classwise(t, p, bins=15):
    e = 0.0
    for c in range(6):
        idx = np.minimum((p[:, c] * bins).astype(int), bins - 1)
        n = np.bincount(idx, minlength=bins); sp = np.bincount(idx, p[:, c], bins); st = np.bincount(idx, t[:, c], bins)
        e += np.abs(sp - st).sum() / len(t)
    return e / 6
om = pd.DataFrame({n: {
    'KLD': K[n].mean(),
    'Brier (vs vote shares)': ((p - yt) ** 2).sum(1).mean(),
    'class-wise ECE': ece_classwise(yt, p),
    'accuracy vs majority vote %': (p.argmax(1) == yti).mean() * 100,
    'macro-F1 %': f1_score(yti, p.argmax(1), average='macro') * 100,
    'macro-AUROC': np.mean([roc_auc_score(oh[:, c], p[:, c]) for c in range(6)]),
    'mean max-probability': p.max(1).mean(),
} for n, p in preds.items()}).T
om.loc['(experts: mean top vote share)'] = {'mean max-probability': yt.max(1).mean()}
om.round(4)

## 3. Per-class metrics

In [ ]:
pc = []
for n, p in preds.items():
    cm = confusion_matrix(yti, p.argmax(1), labels=range(6))
    with np.errstate(invalid='ignore', divide='ignore'):
        rec, prec = np.diag(cm) / cm.sum(1), np.diag(cm) / cm.sum(0)
    f1 = 2 * rec * prec / (rec + prec)
    for c in range(6):
        pc.append({'model': n, 'class': CLASS[c], 'n': int(oh[:, c].sum()), 'AUROC': roc_auc_score(oh[:, c], p[:, c]),
                   'recall %': rec[c] * 100, 'precision %': prec[c] * 100, 'F1 %': f1[c] * 100,
                   'KLD on rows of this class': K[n][yti == c].mean()})
pc = pd.DataFrame(pc)
for metric in ('AUROC', 'recall %', 'precision %', 'F1 %', 'KLD on rows of this class'):
    print(metric); display(pc.pivot(index='class', columns='model', values=metric).loc[CLASS, list(preds)].round(3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.3))
x = np.arange(6); off = np.linspace(-0.3, 0.3, len(preds))
for ax, metric, lab in ((axs[0], 'AUROC', 'AUROC (higher is better)'), (axs[1], 'KLD on rows of this class', 'KLD (lower is better)')):
    piv = pc.pivot(index='class', columns='model', values=metric).loc[CLASS]
    for o, n in zip(off, preds):
        ax.plot(x + o, piv[n].values, 'o', ms=8, color=COL[n], mec=SURF, mew=2, label=n)
    ax.set_xticks(x, [f'{c}\nn={int(oh[:, i].sum())}' for i, c in enumerate(CLASS)]); ax.tick_params(axis='x', colors=INK, length=0)
    ax.set_title(lab, loc='left', fontsize=11, color=INK); ax.grid(axis='x', visible=False)
fig.suptitle('Per class, by majority-vote class of the row', x=0.01, ha='left', fontsize=12, color=INK)
fig.legend(*axs[0].get_legend_handles_labels(), loc='lower center', ncol=len(preds), fontsize=9)
fig.tight_layout(rect=(0, 0.07, 1, 1)); fig.savefig(f'{OUTD}/fig2_per_class.png', dpi=150); plt.close(fig); display(Image(f'{OUTD}/fig2_per_class.png'))

## 4. KLD by number of expert votes

In [ ]:
vb = []
for subset in ('hq', 'all'):
    t_, m_, p_, _ = D[subset]
    edges = [10, 13, 16, 20, 1000] if subset == 'hq' else [1, 2, 3, 6, 10, 16, 1000]
    labs = [f'{a}–{b - 1}' if b < 1000 else f'{a}+' for a, b in zip(edges[:-1], edges[1:])]
    bk = pd.cut(m_.votes, edges, right=False, labels=labs)
    for n, p in p_.items():
        k = pd.Series(kld(t_, p)).groupby(bk.values, observed=True).mean()
        for b_, v in k.items(): vb.append({'subset': subset, 'votes': b_, 'model': n, 'KLD': v, 'n': int((bk == b_).sum())})
vb = pd.DataFrame(vb)
for subset in ('hq', 'all'):
    s = vb[vb.subset == subset]
    if len(s):
        print(subset); display(s.pivot_table(index=['votes', 'n'], columns='model', values='KLD', observed=True, sort=False)[[n for n in NAMES if n in s.model.unique()]].round(4))

## 5. Calibration: predicted probability vs the experts' vote share

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.6))
bins = np.linspace(0, 1, 11)
ax.plot([0, 1], [0, 1], color=INK2, lw=1, ls=(0, (3, 3)))
for n, p in preds.items():
    pf, tf = p.ravel(), yt.ravel(); idx = np.minimum(np.digitize(pf, bins) - 1, 9)
    xs = [pf[idx == i].mean() for i in range(10) if (idx == i).sum() > 50]
    ys = [tf[idx == i].mean() for i in range(10) if (idx == i).sum() > 50]
    ax.plot(xs, ys, '-o', lw=2, ms=6, color=COL[n], mec=SURF, mew=1.5, label=f'{n}  (ECE {om.loc[n, "class-wise ECE"]:.3f})')
ax.set_xlabel('predicted probability (all classes pooled)'); ax.set_ylabel('mean expert vote share')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc='upper left', fontsize=8.5)
ax.set_title('Calibration · on the dashed line = probabilities match vote shares', loc='left', fontsize=10.5, color=INK)
fig.tight_layout(); fig.savefig(f'{OUTD}/fig3_calibration.png', dpi=150); plt.close(fig); display(Image(f'{OUTD}/fig3_calibration.png'))

## 6. Confusion matrices (rows = majority-vote class, row-normalised %)

In [ ]:
n_ = len(preds); fig, axs = plt.subplots(1, n_, figsize=(3.3 * n_, 3.6))
cmap = matplotlib.colors.LinearSegmentedColormap.from_list('blue', ['#f0efec', '#86b6ef', '#2a78d6', '#104281'])
for ax, (n, p) in zip(np.atleast_1d(axs), preds.items()):
    cm = confusion_matrix(yti, p.argmax(1), labels=range(6)).astype(float); cm = cm / cm.sum(1, keepdims=True) * 100
    ax.imshow(cm, cmap=cmap, vmin=0, vmax=100)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f'{cm[i, j]:.0f}', ha='center', va='center', fontsize=7.5, color='white' if cm[i, j] > 55 else INK)
    ax.set_xticks(range(6), CLASS, rotation=60, fontsize=7.5); ax.set_yticks(range(6), CLASS, fontsize=7.5)
    ax.grid(False); ax.set_title(n, fontsize=9.5, color=INK); ax.tick_params(length=0)
fig.supxlabel('predicted class', fontsize=9, color=INK2); fig.tight_layout()
fig.savefig(f'{OUTD}/fig4_confusion.png', dpi=150); plt.close(fig); display(Image(f'{OUTD}/fig4_confusion.png'))

## 7. How different are the models' errors? (blend potential)

In [ ]:
nm = list(preds)
corr = pd.DataFrame([[np.corrcoef(K[a], K[b])[0, 1] for b in nm] for a in nm], index=nm, columns=nm)
agree = pd.DataFrame([[(preds[a].argmax(1) == preds[b].argmax(1)).mean() * 100 for b in nm] for a in nm], index=nm, columns=nm)
best2 = []
for i, a in enumerate(nm):
    for b in nm[i + 1:]:
        g = min((kld(yt, (1 - w) * preds[a] + w * preds[b]).mean(), w) for w in np.linspace(0, 1, 21))
        best2.append({'pair': f'{a} + {b}', 'best arithmetic blend KLD (in-sample)': g[0], 'weight on 2nd': g[1]})
print('correlation of per-row KLD'); display(corr.round(3))
print('argmax agreement %'); display(agree.round(1))
print('pairwise blends, weight tuned on the same rows (optimistic; use blend_5fold.py for the honest number)')
display(pd.DataFrame(best2).sort_values('best arithmetic blend KLD (in-sample)').round(4))

## 8. Each model's own folds

In [ ]:
fk = pd.DataFrame({n: pd.Series(K[n]).groupby(folds[n]).mean() for n in preds}).rename_axis('fold')
fk.loc['mean'] = fk.mean(); fk.loc['std'] = fk.iloc[:5].std(); fk.round(4)

## Save

In [ ]:
kt.to_csv(f'{OUTD}/kld_ci.csv'); om.to_csv(f'{OUTD}/overall_metrics.csv'); pc.to_csv(f'{OUTD}/per_class.csv', index=False)
vb.to_csv(f'{OUTD}/kld_by_votes.csv', index=False); corr.to_csv(f'{OUTD}/error_correlation.csv'); fk.to_csv(f'{OUTD}/per_fold.csv')
def mdt(df):
    try: return df.round(4).to_markdown()
    except ImportError: return '```\n' + df.round(4).to_string() + '\n```'
rep = [f'# HMS model comparison ({pd.Timestamp.today().date()})', f'{len(yt):,} rows with >=10 votes, {len(up):,} patients; reference {REF}.',
       '## KLD (95% patient CI)', mdt(kt), '## Overall metrics', mdt(om),
       '## Per-class AUROC', mdt(pc.pivot(index='class', columns='model', values='AUROC').loc[CLASS, nm]),
       '## Per-class recall %', mdt(pc.pivot(index='class', columns='model', values='recall %').loc[CLASS, nm]),
       '## Per fold', mdt(fk), '## Error correlation', mdt(corr),
       '![kld](fig1_kld.png)\n![per class](fig2_per_class.png)\n![calibration](fig3_calibration.png)\n![confusion](fig4_confusion.png)']
open(f'{OUTD}/report.md', 'w').write('\n\n'.join(rep))
json.dump({'reference': REF, 'models': MODELS, 'n_rows_hq': int(len(yt)), 'kld': kt.reset_index().to_dict('records')},
          open(f'{OUTD}/summary.json', 'w'), indent=2, default=float)
print('saved to', OUTD); print(sorted(os.listdir(OUTD)))